In [2]:
pip install torch torchvision jupyter

Note: you may need to restart the kernel to use updated packages.


In [1]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

import torch
from torchvision import datasets, transforms

print("torch vision is imported")
transform = transforms.ToTensor()

train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

print(len(train_dataset), len(test_dataset))

torch vision is imported


100%|██████████| 9.91M/9.91M [00:01<00:00, 7.63MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 295kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 2.87MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 7.09MB/s]

60000 10000


In [2]:
## Step 2 — Build the model

In [3]:
import torch.nn as nn

model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 64),
    nn.ReLU(),
    nn.Linear(64, 10)
)

print(model)

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=64, bias=True)
  (2): ReLU()
  (3): Linear(in_features=64, out_features=10, bias=True)
)


In [ ]:
## Step 3 - Training


In [6]:
import torch
from torch.utils.data import DataLoader

# dataloader
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 2

for epoch in range(epochs):
    for images, labels in train_loader:
        
        # forward
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    print(f"Epoch {epoch+1}, Loss: {loss.item()}")

Epoch 1, Loss: 0.18623480200767517
Epoch 2, Loss: 0.1630668044090271


In [8]:
##Step 4 — Test the model

In [9]:
test_loader = DataLoader(test_dataset, batch_size=64)

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print("Accuracy:", accuracy)

Accuracy: 95.27


In [10]:
## Step 5 — Extract weights

In [12]:
import numpy as np

W1 = model[1].weight.detach().numpy()
b1 = model[1].bias.detach().numpy()

W2 = model[3].weight.detach().numpy()
b2 = model[3].bias.detach().numpy()

print(W1.shape, b1.shape)
print(W2.shape, b2.shape)

np.save("W1.npy", W1)
np.save("b1.npy", b1)
np.save("W2.npy", W2)
np.save("b2.npy", b2)

(64, 784) (64,)
(10, 64) (10,)


In [13]:
## Step 6 — Manual forward pass (NumPy)

In [14]:
def forward_numpy(x, W1, b1, W2, b2):
    h = np.dot(W1, x) + b1
    h = np.maximum(h, 0)   # ReLU
    y = np.dot(W2, h) + b2
    return y

# get one sample
image, label = test_dataset[0]

# flatten image (28x28 → 784)
x = image.view(-1).numpy()

# PyTorch output
with torch.no_grad():
    torch_output = model(image.unsqueeze(0))
    torch_pred = torch.argmax(torch_output).item()

# NumPy output
numpy_output = forward_numpy(x, W1, b1, W2, b2)
numpy_pred = np.argmax(numpy_output)

print("Label:", label)
print("PyTorch prediction:", torch_pred)
print("NumPy prediction:", numpy_pred)

Label: 7
PyTorch prediction: 7
NumPy prediction: 7


In [15]:
## Step 8 — Measure CPU inference time

In [18]:
import time

# prepare multiple samples
inputs = []
for i in range(1000):
    image, _ = test_dataset[i]
    x = image.view(-1).numpy()
    inputs.append(x)

# measure time
start = time.time()

for x in inputs:
    forward_numpy(x, W1, b1, W2, b2)

end = time.time()

print("CPU time for 1000 samples:", end - start)


CPU time for 1000 samples: 0.0032231807708740234


In [19]:
print(W1.dtype, W2.dtype)

float32 float32


In [20]:
## Step 10 — Prepare data for FPGA

In [21]:
# pick one sample
image, label = test_dataset[0]

# flatten
x = image.view(-1).numpy().astype(np.float32)

print("x shape:", x.shape)
print("W1 shape:", W1.shape)

x shape: (784,)
W1 shape: (64, 784)
